# Extension 2: Cross-Dataset Structural Robustness

**Objective:** Prove that regularized Bilinear MLPs learn universal geometric shapes (e.g., "circularity") rather than dataset-specific pixel artifacts.

## Key Hypothesis

If regularization encourages low-rank structure, and low-rank structure captures "universal" shape features, then:
1. A regularized model should perform well on a **different digit dataset** (USPS)
2. A regularized model should classify EMNIST letters by their *shape* (O→0, I→1, etc.)
3. The eigenvector subspaces for similar shapes should overlap significantly

## Methodology

### Center-of-Mass Normalization
**Critical constraint:** Bilinear MLPs are NOT translation invariant. To fairly compare datasets, we normalize all images by center-of-mass.

### Three-Step Validation

**Step 1: USPS Transfer Test**
- Train baseline (no noise) and regularized (σ=0.15) models on MNIST
- Evaluate both on USPS (different digit dataset, same classes 0-9)
- Success: High accuracy proves eigenvectors capture universal digit features

**Step 2: Semantic Confusion Test**
- Evaluate both MNIST-trained models on EMNIST letters 'O', 'I', 'Z', 'S', 'B'
- Success: Regularized model classifies by shape (O→0, I→1, etc.)

**Step 3: Subspace Geometry Metric**
- Train EMNIST model and extract eigenvectors
- Compare MNIST '0' eigenvectors with EMNIST 'O' eigenvectors
- Use Principal Angles to quantify subspace overlap

In [ ]:
# Setup
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "bilinear-decomposition-main"))

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Our modules
from src.data.mnist import MNIST
from src.data.emnist import (
    load_emnist_letters_normalized,
    LETTER_DIGIT_SIMILARITY,
    EMNIST_CLASS_NAMES,
)
from src.data.usps import load_usps_normalized
from src.vision.subspace import compute_subspace_overlap, principal_angles
from src.vision.spectral import effective_rank
from src.plot_utils.extension2 import (
    plot_usps_transfer_comparison,
    plot_semantic_confusion_distributions,
    plot_subspace_overlap_comparison,
    plot_eigenspectra_side_by_side,
)
from src.utils import get_device

# Set device
device = get_device()
print(f"Using device: {device}")

# Paths
RESULTS_DIR = PROJECT_ROOT / "results/extension2"
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
FIGURE_DIR = PROJECT_ROOT / "Report/figures/extension2"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Plotting style
plt.rcParams['figure.dpi'] = 150

## 1. Visualize Center-of-Mass Normalization

First, let's verify that our normalization pipeline works correctly.

In [ ]:
# Load normalized datasets
print("Loading normalized datasets...")
# MNIST: Use class directly with apply_com=True for CoM normalization
mnist_train = MNIST(train=True, device='cpu', apply_com=True)
mnist_test = MNIST(train=False, device='cpu', apply_com=True)
# EMNIST and USPS: Use convenience functions
emnist_train, emnist_test = load_emnist_letters_normalized(device='cpu', apply_com=True)
usps_train, usps_test = load_usps_normalized(device='cpu', apply_com=True)

print(f"MNIST test: {len(mnist_test)} samples, {mnist_test.n_classes} classes")
print(f"USPS test: {len(usps_test)} samples (same 10 digit classes)")
print(f"EMNIST test: {len(emnist_test)} samples, {emnist_test.n_classes} letter classes")

In [ ]:
# Visualize sample images from each dataset
fig, axes = plt.subplots(3, 10, figsize=(15, 6))

# MNIST digits 0-9
for digit in range(10):
    mask = mnist_test.y == digit
    idx = mask.nonzero()[0][0]
    img = mnist_test.x[idx].squeeze().numpy()
    axes[0, digit].imshow(img, cmap='gray')
    axes[0, digit].axis('off')
    axes[0, digit].set_title(f'{digit}')

# USPS digits 0-9 (same classes as MNIST, different dataset)
for digit in range(10):
    mask = usps_test.y == digit
    if mask.sum() > 0:
        idx = mask.nonzero()[0][0]
        img = usps_test.x[idx].squeeze().numpy()
    else:
        img = np.zeros((28, 28))
    axes[1, digit].imshow(img, cmap='gray')
    axes[1, digit].axis('off')
    axes[1, digit].set_title(f'{digit}')

# EMNIST letters that should look like digits
letters_of_interest = ['O', 'I', 'Z', 'S', 'B', 'A', 'C', 'E', 'G', 'H']
for i, letter in enumerate(letters_of_interest):
    letter_idx = ord(letter) - ord('A')
    mask = emnist_test.y == letter_idx
    idx = mask.nonzero()[0][0]
    img = emnist_test.x[idx].squeeze().numpy()
    axes[2, i].imshow(img, cmap='gray')
    axes[2, i].axis('off')
    expected = LETTER_DIGIT_SIMILARITY.get(letter, '?')
    axes[2, i].set_title(f'{letter}→{expected}')

axes[0, 0].text(-0.5, 0.5, 'MNIST', transform=axes[0, 0].transAxes, 
               ha='right', va='center', fontsize=12, fontweight='bold', rotation=90)
axes[1, 0].text(-0.5, 0.5, 'USPS', transform=axes[1, 0].transAxes,
               ha='right', va='center', fontsize=12, fontweight='bold', rotation=90)
axes[2, 0].text(-0.5, 0.5, 'EMNIST', transform=axes[2, 0].transAxes,
               ha='right', va='center', fontsize=12, fontweight='bold', rotation=90)

plt.suptitle('Center-of-Mass Normalized Images: Three Datasets', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'ext2_normalized_samples.pdf', bbox_inches='tight')
plt.show()

## 2. Load Pre-trained Models

If checkpoints exist from running `evaluate_extension2.py --full-pipeline`, load them.

In [ ]:
# Check for existing checkpoints
checkpoint_files = list(CHECKPOINT_DIR.glob("*.pt")) if CHECKPOINT_DIR.exists() else []

if len(checkpoint_files) >= 3:
    print(f"Found {len(checkpoint_files)} checkpoints:")
    for f in checkpoint_files:
        print(f"  - {f.name}")
    
    # Load checkpoints
    ckpt_baseline = torch.load(CHECKPOINT_DIR / "mnist_baseline_seed42.pt", map_location='cpu')
    ckpt_regularized = torch.load(CHECKPOINT_DIR / "mnist_regularized_seed42.pt", map_location='cpu')
    ckpt_emnist = torch.load(CHECKPOINT_DIR / "emnist_regularized_seed42.pt", map_location='cpu')
    
    # Extract eigenvalues and eigenvectors
    vals_baseline = ckpt_baseline['eigenvalues']
    vecs_baseline = ckpt_baseline['eigenvectors']
    vals_regularized = ckpt_regularized['eigenvalues']
    vecs_regularized = ckpt_regularized['eigenvectors']
    vals_emnist = ckpt_emnist['eigenvalues']
    vecs_emnist = ckpt_emnist['eigenvectors']
    
    print("\nLoaded eigenvalues/eigenvectors:")
    print(f"  Baseline MNIST: {vals_baseline.shape}")
    print(f"  Regularized MNIST: {vals_regularized.shape}")
    print(f"  EMNIST: {vals_emnist.shape}")
else:
    print("No checkpoints found. Run the following command first:")
    print("  python src/evaluate_extension2.py --full-pipeline --epochs 100")
    print("\nOr submit on Snellius:")
    print("  sbatch jobs/extension2_full.job")
    
    vals_baseline = vals_regularized = vals_emnist = None
    vecs_baseline = vecs_regularized = vecs_emnist = None

## 3. Step 1: USPS Transfer Test Results

Test if MNIST-trained models generalize to USPS digits (same classes, different dataset).

In [ ]:
# Load USPS transfer results
usps_results_path = RESULTS_DIR / "usps_transfer_results.json"

if usps_results_path.exists():
    with open(usps_results_path) as f:
        usps_results = json.load(f)
    
    print("USPS Transfer Test Results")
    print("=" * 60)
    print("(High accuracy = learned universal digit features, not MNIST artifacts)")
    
    # Overall comparison
    print(f"\nOverall USPS Accuracy:")
    print(f"  Baseline:    {usps_results['conclusion']['baseline_accuracy']:.2%}")
    print(f"  Regularized: {usps_results['conclusion']['regularized_accuracy']:.2%}")
    print(f"  Improvement: {usps_results['conclusion']['improvement']:+.2%}")
    
    # Per-class comparison table
    table_data = []
    for digit in range(10):
        base_acc = usps_results['baseline']['per_class_accuracy'].get(str(digit), {}).get('accuracy', 0)
        reg_acc = usps_results['regularized']['per_class_accuracy'].get(str(digit), {}).get('accuracy', 0)
        table_data.append({
            'Digit': digit,
            'Baseline': f"{base_acc:.2%}",
            'Regularized': f"{reg_acc:.2%}",
            'Δ': f"{reg_acc - base_acc:+.2%}"
        })
    
    df_usps = pd.DataFrame(table_data)
    display(df_usps)
else:
    print("USPS results not found. Run the full pipeline first.")

In [ ]:
# Visualize USPS transfer accuracy
if usps_results_path.exists():
    fig = plot_usps_transfer_comparison(
        usps_results, 
        save_path=FIGURE_DIR / 'ext2_usps_transfer.pdf'
    )
    plt.show()

## 4. Step 2: Semantic Confusion Test Results

Analyze how baseline vs regularized models classify EMNIST letters.

In [ ]:
# Load semantic confusion results
semantic_results_path = RESULTS_DIR / "semantic_confusion_results.json"

if semantic_results_path.exists():
    with open(semantic_results_path) as f:
        semantic_results = json.load(f)
    
    print("Semantic Confusion Test Results")
    print("=" * 60)
    
    # Create comparison table
    letters = ['O', 'I', 'Z', 'S', 'B']
    table_data = []
    
    for letter in letters:
        baseline = semantic_results['baseline'][letter]
        regularized = semantic_results['regularized'][letter]
        expected = LETTER_DIGIT_SIMILARITY[letter]
        
        table_data.append({
            'Letter': letter,
            'Expected Digit': expected,
            'Baseline Pred': baseline['modal_prediction'],
            'Baseline Conf': f"{baseline['modal_confidence']:.2f}",
            'Regularized Pred': regularized['modal_prediction'],
            'Regularized Conf': f"{regularized['modal_confidence']:.2f}",
            'Baseline ✓': '✓' if baseline['modal_prediction'] == expected else '✗',
            'Regularized ✓': '✓' if regularized['modal_prediction'] == expected else '✗',
        })
    
    df_semantic = pd.DataFrame(table_data)
    display(df_semantic)
    
    print(f"\nBaseline accuracy: {semantic_results['conclusion']['baseline_accuracy']:.2%}")
    print(f"Regularized accuracy: {semantic_results['conclusion']['regularized_accuracy']:.2%}")
    print(f"Improvement: {semantic_results['conclusion']['improvement']:+.2%}")
else:
    print("Semantic results not found. Run the full pipeline first.")

In [ ]:
# Visualize prediction distributions
if semantic_results_path.exists():
    fig = plot_semantic_confusion_distributions(
        semantic_results,
        LETTER_DIGIT_SIMILARITY,
        save_path=FIGURE_DIR / 'ext2_semantic_confusion.pdf'
    )
    plt.show()

## 5. Step 3: Subspace Geometry Analysis

Compare eigenvector subspaces between MNIST digits and EMNIST letters.

In [ ]:
# Load subspace geometry results
subspace_results_path = RESULTS_DIR / "subspace_geometry_results.json"

if subspace_results_path.exists():
    with open(subspace_results_path) as f:
        subspace_results = json.load(f)
    
    print("Subspace Geometry Test Results")
    print("=" * 60)
    
    # Display expected pairs
    print("\nExpected shape-similar pairs:")
    for pair, metrics in subspace_results['expected_pairs'].items():
        print(f"  {pair}: mean_cos={metrics['mean_cos']:.4f}, "
              f"grassmann={metrics['grassmann']:.4f}, "
              f"projection={metrics['projection']:.4f}")
    
    print(f"\nRandom baseline: {subspace_results['random_baseline']['mean']:.4f} ± {subspace_results['random_baseline']['std']:.4f}")
    print(f"\nExpected pairs mean: {subspace_results['summary']['expected_mean']:.4f}")
    print(f"Ratio (expected/random): {subspace_results['summary']['ratio']:.2f}x")
    print(f"Conclusion: {subspace_results['summary']['conclusion']}")
else:
    print("Subspace results not found. Run the full pipeline first.")

# Visualize subspace overlaps
if subspace_results_path.exists():
    fig = plot_subspace_overlap_comparison(
        subspace_results,
        save_path=FIGURE_DIR / 'ext2_subspace_overlap.pdf'
    )
    plt.show()

In [ ]:
## 6. Eigenspectra Comparison (Optional)

if vals_baseline is not None and vals_regularized is not None:
    fig = plot_eigenspectra_side_by_side(
        vals_baseline,
        vals_regularized,
        save_path=FIGURE_DIR / 'ext2_eigenspectra.pdf'
    )
    plt.show()

## 7. Conclusions

### Key Findings

**Step 1: USPS Transfer Test**
- [ ] Regularized model achieves high accuracy on USPS (different digit dataset)
- [ ] Baseline model shows lower transfer performance
- [ ] This proves eigenvectors capture universal digit features, not MNIST-specific artifacts

**Step 2: Semantic Confusion Test**
- [ ] Regularized model correctly maps visually similar letters to digits (O→0, I→1)
- [ ] Baseline model fails to recognize shape similarity
- [ ] This proves regularization encourages learning of universal geometric features

**Step 3: Subspace Geometry Metric**
- [ ] Expected pairs (0-O, 1-I, etc.) show significantly higher overlap than random pairs
- [ ] Principal angles confirm that eigenvector subspaces capture similar mechanisms
- [ ] Low-rank structure corresponds to universal shape features, not dataset artifacts

### Interpretation

The three-step validation provides strong evidence that:

1. **Step 1 (USPS)**: High accuracy on a different digit dataset proves the eigenvectors aren't memorizing MNIST-specific pixel patterns—they capture universal digit geometry.

2. **Step 2 (EMNIST Letters)**: Correct letter-to-digit mapping shows the model learned shape-based representations that transfer beyond digits to semantically similar characters.

3. **Step 3 (Subspace Overlap)**: High eigenvector subspace overlap for similar shapes (0-O, 1-I) mathematically confirms that the learned mechanisms are truly universal.

**Conclusion**: Regularization → Low Rank → Universal Features. The low-rank structure induced by regularization doesn't just compress the representation—it forces the model to learn generalizable geometric primitives that transfer across datasets.

### Figures Generated

- `ext2_normalized_samples.pdf` - Center-of-mass normalized samples (3 datasets)
- `ext2_usps_transfer.pdf` - USPS transfer test results
- `ext2_semantic_confusion.pdf` - Letter classification distributions
- `ext2_subspace_overlap.pdf` - Eigenvector subspace similarity
- `ext2_eigenspectra.pdf` - Eigenvalue spectra comparison